# Chẩn đoán bệnh lá từ ảnh chụp điện thoại

**Sổ Nghiên Cứu AI, dự án mẫu 1/10.** Học chuyển giao với mạng nơ-ron tích chập, so với mô hình đối chứng đơn giản.

**Câu hỏi nghiên cứu:** Mô hình học chuyển giao nhỏ có giữ được độ chính xác khi chuyển từ ảnh có sẵn sang ảnh chụp ngoài ruộng không?

**Giả thuyết:** Huấn luyện thêm với ảnh thực địa tự chụp tăng F1 trên ảnh thực địa ít nhất 15 điểm phần trăm.

### Cách dùng notebook này
1. Trên Colab, chọn **Runtime → Change runtime type → T4 GPU** nếu có (không bắt buộc).
2. Chạy lần lượt từng ô bằng **Shift + Enter**. Lần đầu có thể mất vài phút để cài thư viện và tải dữ liệu.
3. Các ô **Việc của em** là chỗ em tự làm tiếp để biến dự án mẫu thành nghiên cứu của riêng mình.

Dữ liệu luyện tập mặc định là bộ **beans** (bệnh lá đậu, 3 lớp) trên Hugging Face, vì nó tải được ngay. Khi làm thật, em thay bằng ảnh lá lúa của em theo hướng dẫn ở ô dữ liệu.

> **Liêm chính khoa học.** Đây là mã mẫu để luyện tập. Nếu dùng cho dự án dự thi, hãy ghi rõ trong sổ nghiên cứu và báo cáo đoạn mã nào lấy từ notebook này, đoạn nào em tự viết hoặc sửa. Dữ liệu dự thi phải do em thu thập hoặc lấy từ nguồn công khai có ghi nguồn.

In [ ]:
%pip install -q datasets onnx

In [ ]:
import os, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 110
def tieu_de(s): print("\n" + "=" * 8, s, "=" * 8)

import torch, torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageDraw
_ = torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Thiết bị:", device, "| torch", torch.__version__)
EPOCHS = int(os.environ.get("SNC_EPOCHS", 5))   # số vòng huấn luyện, tăng lên khi có GPU

## 1. Dữ liệu

In [ ]:
# ---- NGUỒN DỮ LIỆU ----
# Cách 1 (dùng khi làm dự án thật): ảnh của em, xếp theo thư mục  data/<ten_lop>/<anh>.jpg
#   Trên Colab: kéo thả file zip vào khung Files bên trái rồi chạy  !unzip -q data.zip
DATA_DIR = "data"
# Cách 2 (mặc định để luyện tập): bộ dữ liệu công khai trên Hugging Face
HF_DATASET = "beans"   # bệnh lá đậu: angular_leaf_spot, bean_rust, healthy
# Cách 3 (dự phòng khi không có mạng): ảnh giả lập, chỉ để kiểm tra quy trình chạy được

def tai_thu_muc(d):
    lop = sorted(c for c in os.listdir(d) if os.path.isdir(os.path.join(d, c)))
    mau = [(os.path.join(d, c, f), i) for i, c in enumerate(lop)
           for f in sorted(os.listdir(os.path.join(d, c))) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    return mau, lop

def tai_hf(ten, toi_da=1500):
    from datasets import load_dataset
    ds = load_dataset(ten)
    tach = [k for k in ds.keys()]
    feat = ds[tach[0]].features
    cot_nhan = [k for k in feat if k not in ("image", "img", "image_file_path")][0]
    cot_anh = "image" if "image" in feat else [k for k in feat if k != cot_nhan][0]
    lop = feat[cot_nhan].names
    mau = []
    for t in tach:
        for r in ds[t]:
            mau.append((r[cot_anh].convert("RGB"), r[cot_nhan]))
    random.shuffle(mau)
    return mau[:toi_da], lop

def tao_gia_lap(lop=('dom_la', 'gi_sat', 'khoe'), n=60):
    mau = []
    for i, _ in enumerate(lop):
        for _ in range(n):
            im = Image.new("RGB", (128, 128), (70 + 20 * i, 140 - 15 * i, 60))
            dr = ImageDraw.Draw(im)
            for _ in range(3 + 4 * i):
                x, y = np.random.randint(0, 120, 2); r = np.random.randint(3, 6 + 2 * i)
                dr.ellipse([x, y, x + r, y + r], fill=(150 + 30 * i, 110, 40))
            mau.append((im, i))
    random.shuffle(mau)
    return mau, list(lop)

if os.path.isdir(DATA_DIR) and len(os.listdir(DATA_DIR)) > 1:
    MAU, LOP = tai_thu_muc(DATA_DIR); NGUON = "ảnh của em"
else:
    try:
        MAU, LOP = tai_hf(HF_DATASET); NGUON = "Hugging Face: " + HF_DATASET
    except Exception as e:
        print("Không tải được dữ liệu công khai (", type(e).__name__, "). Dùng ảnh giả lập để chạy thử quy trình.")
        MAU, LOP = tao_gia_lap(); NGUON = "GIẢ LẬP (không dùng để báo cáo)"
print("Nguồn:", NGUON, "| số ảnh:", len(MAU), "| các lớp:", LOP)
print(pd.Series([LOP[y] for _, y in MAU]).value_counts())

In [ ]:
tieu_de("Xem thử vài ảnh")
fig, ax = plt.subplots(1, 6, figsize=(12, 2.4))
for a, (x, y) in zip(ax, MAU[:6]):
    a.imshow(Image.open(x).convert("RGB") if isinstance(x, str) else x); a.set_title(LOP[y], fontsize=8); a.axis("off")
plt.show()

## 2. Chia dữ liệu
Tập kiểm tra được cất đi, chỉ dùng **một lần** ở cuối. Mọi lựa chọn (số vòng, tham số) dựa trên tập kiểm định.

In [ ]:
# Chia 70% huấn luyện, 15% kiểm định, 15% kiểm tra, giữ tỉ lệ các lớp (stratified).
# CẢNH BÁO RÒ RỈ DỮ LIỆU: nếu em chụp nhiều ảnh của CÙNG một chiếc lá / một vật,
# phải chia theo vật (group split), không chia theo ảnh. Xem Việc của em bên dưới.
from sklearn.model_selection import train_test_split
y_all = [y for _, y in MAU]
tr, tmp = train_test_split(MAU, test_size=0.3, stratify=y_all, random_state=SEED)
va, te = train_test_split(tmp, test_size=0.5, stratify=[y for _, y in tmp], random_state=SEED)
print(f"Huấn luyện {len(tr)} | kiểm định {len(va)} | kiểm tra {len(te)}")

## 3. Mô hình đối chứng
Không có đối chứng thì con số 90% không nói lên điều gì. Ở đây đối chứng là hồi quy logistic trên biểu đồ màu của ảnh.

In [ ]:
tieu_de("Mô hình đối chứng: hồi quy logistic trên đặc trưng màu")
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
def mo_anh(x): return (Image.open(x) if isinstance(x, str) else x).convert("RGB").resize((96, 96))
def dac_trung_mau(x):
    a = np.asarray(mo_anh(x).convert("HSV"))
    return np.concatenate([np.histogram(a[..., c], bins=16, range=(0, 255), density=True)[0] for c in range(3)])
Xtr = np.array([dac_trung_mau(x) for x, _ in tr]); ytr = [y for _, y in tr]
Xte = np.array([dac_trung_mau(x) for x, _ in te]); yte = [y for _, y in te]
base = LogisticRegression(max_iter=3000).fit(Xtr, ytr)
p_base = base.predict(Xte)
KQ = {"Đối chứng: logistic + màu": (accuracy_score(yte, p_base), f1_score(yte, p_base, average="macro"))}
print("Độ chính xác %.3f | F1 trung bình theo lớp %.3f" % KQ["Đối chứng: logistic + màu"])

## 4. Mô hình đề xuất

In [ ]:
tieu_de("Mô hình đề xuất: học chuyển giao MobileNetV3")
IMG = 224
tf_train = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), transforms.ColorJitter(0.2, 0.2, 0.2),   # tăng cường dữ liệu
    transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
tf_test = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
class TapAnh(Dataset):
    def __init__(s, mau, tf): s.mau, s.tf = mau, tf
    def __len__(s): return len(s.mau)
    def __getitem__(s, i):
        x, y = s.mau[i]; return s.tf(mo_anh(x).resize((IMG, IMG))), y
dl = lambda m, tf, sh: DataLoader(TapAnh(m, tf), batch_size=32, shuffle=sh, num_workers=0)
dl_tr, dl_va, dl_te = dl(tr, tf_train, True), dl(va, tf_test, False), dl(te, tf_test, False)

def tao_mo_hinh(so_lop, hoc_chuyen_giao=True):
    try:
        w = torchvision.models.MobileNet_V3_Small_Weights.DEFAULT if hoc_chuyen_giao else None
        m = torchvision.models.mobilenet_v3_small(weights=w)
    except Exception as e:
        print("Không tải được trọng số huấn luyện sẵn (", type(e).__name__, "), dùng khởi tạo ngẫu nhiên.")
        m = torchvision.models.mobilenet_v3_small(weights=None)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, so_lop)
    return m.to(device)

def du_doan(m, loader):
    m.eval(); P, Y = [], []
    with torch.no_grad():
        for x, y in loader: P += m(x.to(device)).argmax(1).cpu().tolist(); Y += y.tolist()
    return np.array(Y), np.array(P)

def huan_luyen(m, so_vong=EPOCHS):
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4); lossf = nn.CrossEntropyLoss()
    lich_su = []
    for ep in range(so_vong):
        m.train(); t0 = time.time()
        for x, y in dl_tr:
            opt.zero_grad(); l = lossf(m(x.to(device)), y.to(device)); l.backward(); opt.step()
        yv, pv = du_doan(m, dl_va); f = f1_score(yv, pv, average="macro"); lich_su.append(f)
        print(f"Vòng {ep+1}: F1 kiểm định = {f:.3f}  ({time.time()-t0:.0f} giây)")
    return lich_su

model = tao_mo_hinh(len(LOP))
ls = huan_luyen(model)
yt, pt = du_doan(model, dl_te)
KQ["Học chuyển giao MobileNetV3"] = (accuracy_score(yt, pt), f1_score(yt, pt, average="macro"))
print("\nTRÊN TẬP KIỂM TRA (chỉ dùng một lần, ở cuối):")
print(classification_report(yt, pt, target_names=LOP, digits=3))

## 5. Đánh giá

In [ ]:
tieu_de("So sánh và phân tích lỗi")
print(pd.DataFrame(KQ, index=["Độ chính xác", "F1 trung bình"]).T.round(3))
ConfusionMatrixDisplay(confusion_matrix(yt, pt), display_labels=LOP).plot(cmap="Blues", xticks_rotation=45)
plt.title("Ma trận nhầm lẫn trên tập kiểm tra"); plt.show()
sai = [i for i in range(len(te)) if yt[i] != pt[i]][:8]
if sai:
    fig, ax = plt.subplots(1, len(sai), figsize=(2 * len(sai), 2.4))
    for a, i in zip(np.atleast_1d(ax), sai):
        a.imshow(mo_anh(te[i][0])); a.set_title(f"thật: {LOP[yt[i]]}\nđoán: {LOP[pt[i]]}", fontsize=7); a.axis("off")
    plt.suptitle("Các ca mô hình đoán sai: xem kỹ, ghi nhận xét vào sổ"); plt.show()
tieu_de("Thời gian suy luận một ảnh trên CPU")
mc = model.to("cpu").eval(); x = torch.randn(1, 3, IMG, IMG)
with torch.no_grad():
    for _ in range(3): mc(x)
    t0 = time.time()
    for _ in range(20): mc(x)
print(f"{(time.time()-t0)/20*1000:.1f} ms mỗi ảnh")
model = model.to(device)

## 6. Thử nghiệm đổi miền ảnh (phần tạo tính mới)
Đặt ảnh tự chụp ngoài ruộng vào thư mục `field/<ten_lop>/` với **cùng tên lớp**. Ô dưới đo mô hình trên ảnh thực địa.

In [ ]:
if os.path.isdir("field"):
    mau_f, lop_f = tai_thu_muc("field")
    assert lop_f == LOP, "Tên thư mục lớp trong field/ phải giống hệt các lớp huấn luyện"
    yf, pf = du_doan(model, dl(mau_f, tf_test, False))
    print("Trên ảnh thực địa: F1 = %.3f (so với %.3f trên tập kiểm tra)" % (f1_score(yf, pf, average="macro"), KQ["Học chuyển giao MobileNetV3"][1]))
else:
    print("Chưa có thư mục field/. Hãy chụp ảnh thực địa rồi chạy lại ô này.")

In [ ]:
# Lưu mô hình để dùng trên điện thoại hoặc máy tính nhúng (ONNX chạy được trên nhiều nền tảng)
try:
    torch.onnx.export(model.to("cpu").eval(), torch.randn(1, 3, IMG, IMG), "mo_hinh.onnx", input_names=["anh"], output_names=["lop"], dynamo=False)
    print("Đã lưu mo_hinh.onnx, kích thước %.1f MB" % (os.path.getsize("mo_hinh.onnx") / 1e6))
except Exception as e:
    print("Chưa xuất được ONNX:", e)
finally:
    model.to(device)

## Việc của em
1. **Dữ liệu thật:** thay bằng ảnh lá lúa (đạo ôn, bạc lá, đốm nâu, khỏe). Ghi vào sổ: ngày, nơi chụp, điện thoại, ai gán nhãn.
2. **Chống rò rỉ:** nếu một chiếc lá có nhiều ảnh, đặt tên file `la012_a.jpg, la012_b.jpg` rồi sửa ô chia dữ liệu dùng `GroupShuffleSplit` theo mã lá.
3. **Lặp lại:** đổi `SEED` 5 lần, ghi F1 mỗi lần, dán vào công cụ *Kiểm tra kết quả* trên trang Sổ Nghiên Cứu AI để xem khác biệt có thật không.
4. **Tính mới:** đo mức sụt giảm khi đổi miền ảnh, rồi thử thêm 100, 200, 300 ảnh thực địa vào tập huấn luyện. Vẽ đồ thị F1 theo số ảnh thêm vào.